In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

model = nn.Linear(10, 2)
inputs = torch.randn(4, 10)
targets = torch.randn(4, 2)

criterion  = nn.MSELoss()

optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum=0.9)


optimizer.zero_grad()
loss = criterion(model(inputs), targets)
loss.backward()
optimizer.step()

In [3]:
import torch

def narrow_valley_loss(w):
    """
    A simple 2D loss function creating a narrow valley.
    High curvature on w[0] (steep walls), low curvature on w[1] (flat floor).
    """
    return (100 * w[0]**2) + (w[1] - 5)**2

# ---------------------------------------------------------
# 1. Implementation from scratch (The exact math)
# ---------------------------------------------------------
w_custom = torch.tensor([-2.0, -2.0], requires_grad=True)
velocity = torch.zeros_like(w_custom) # Initialize velocity v_0 = 0

learning_rate = 0.01
momentum_coeff = 0.9

print("Custom Momentum Steps:")
for step in range(5):
    loss = narrow_valley_loss(w_custom)
    loss.backward()
    
    with torch.no_grad(): # Do not track gradient operations for the update
        # Math: v_{t+1} = \mu v_t + \nabla L(w_t)
        velocity = (momentum_coeff * velocity) + w_custom.grad
        
        # Math: w_{t+1} = w_t - \eta v_{t+1}
        w_custom -= learning_rate * velocity
        
        # Clear gradient for the next step (optimizer.zero_grad())
        w_custom.grad.zero_()
        
    print(f"Step {step+1} | w: {w_custom.data.numpy().round(4)} | v: {velocity.numpy().round(4)}")

# ---------------------------------------------------------
# 2. PyTorch Native Implementation
# ---------------------------------------------------------
w_native = torch.tensor([-2.0, -2.0], requires_grad=True)

# PyTorch encapsulates the velocity state internally
optimizer = torch.optim.SGD([w_native], lr=learning_rate, momentum=momentum_coeff)

print("\nPyTorch Native Steps:")
for step in range(5):
    optimizer.zero_grad()
    loss = narrow_valley_loss(w_native)
    loss.backward()
    optimizer.step()
    
    print(f"Step {step+1} | w: {w_native.data.numpy().round(4)}")

Custom Momentum Steps:
Step 1 | w: [ 2.   -1.86] | v: [-400.  -14.]
Step 2 | w: [ 1.6    -1.5968] | v: [ 40.   -26.32]
Step 3 | w: [-1.96  -1.228] | v: [356.     -36.8816]
Step 4 | w: [-1.244  -0.7715] | v: [-71.6    -45.6494]
Step 5 | w: [ 1.8884 -0.2452] | v: [-313.24    -52.6274]

PyTorch Native Steps:
Step 1 | w: [ 2.   -1.86]
Step 2 | w: [ 1.6    -1.5968]
Step 3 | w: [-1.96  -1.228]
Step 4 | w: [-1.244  -0.7715]
Step 5 | w: [ 1.8884 -0.2452]


In [1]:
import torchvision.transforms as T

# Defines a random affine transform for data augmentation
affine_augmenter = T.RandomAffine(
    degrees=(-15, 15),      # Rotation
    translate=(0.1, 0.1),   # Horizontal/Vertical shifts
    scale=(0.9, 1.1),       # Zoom in/out
    shear=10                # Skew angle
)

# Apply to a batch of images
augmented_batch = affine_augmenter(image_batch)

RuntimeError: operator torchvision::nms does not exist